In [1]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\envs\qwen3vl\python.exe


In [2]:
import torch
import transformers
from qwen_vl_utils import process_vision_info

print(torch.__version__)
print(transformers.__version__)
print("qwen3vl 환경 정상")

c:\ProgramData\anaconda3\envs\qwen3vl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.7.1+cu118
5.13.0.dev0
qwen3vl 환경 정상


In [3]:
import os
import re
import ast
import json
import numpy as np
import torch
from PIL import Image
from IPython.display import display
from transformers import (
    CLIPProcessor,
    CLIPModel,
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info


# =========================
# 0. 경로 / 설정
# =========================

BASE_DIR = "VLM_RGA"
DATABASE_FILE = os.path.join(BASE_DIR, "rag_database.npy")
QUERY_DIR = os.path.join(BASE_DIR, "dataset", "기준 사진")
TAGS_FILE = os.path.join(BASE_DIR, "tags.txt")

TOP_K = 10
MIN_FILTER_MATCH = 3

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


# =========================
# 1. 경로 보정
# =========================

def resolve_path(path):
    if os.path.exists(path):
        return path

    alt_path = path.replace("VLM_RGA\\", "").replace("VLM_RGA/", "")
    if os.path.exists(alt_path):
        return alt_path

    raise FileNotFoundError(f"경로를 찾을 수 없음: {path}")


def get_query_image_path():
    valid_ext = (".jpg", ".jpeg", ".png", ".webp")
    query_dir = resolve_path(QUERY_DIR)

    images = [
        os.path.join(query_dir, f)
        for f in os.listdir(query_dir)
        if f.lower().endswith(valid_ext)
    ]

    if not images:
        raise FileNotFoundError(f"기준 사진이 없습니다: {query_dir}")

    return images[0]


# =========================
# 2. RAG DB / 태그 후보 불러오기
# =========================

def load_database():
    return np.load(resolve_path(DATABASE_FILE), allow_pickle=True)


def collect_all_tags(database):
    all_tags = set()
    for item in database:
        for tag in item["tags"]:
            tag = str(tag).strip()
            if tag:
                all_tags.add(tag)
    return sorted(all_tags)


# =========================
# 3. CLIP 이미지 임베딩
# =========================

print("CLIP 모델 로드 중...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP 모델 로드 완료")


def get_image_embedding(image_path):
    image_path = resolve_path(image_path)
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)

    if not isinstance(features, torch.Tensor):
        features = features.pooler_output

    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()[0]


def cosine_similarity(a, b):
    return float(np.dot(a, b))


# =========================
# 4. Qwen-VL 모델 로드
# =========================

print("Qwen-VL 모델 로드 중...")
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model.eval()
print("Qwen-VL 모델 로드 완료")


# =========================
# 5. Qwen3-VL이 tags.txt 후보 중 해당 태그 선택
# =========================

def parse_json_list_from_text(text):
    text = text.strip()

    # ```json ... ``` 제거
    text = re.sub(r"^```json", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    # JSON object 형태: {"selected_tags": [...]}
    try:
        data = json.loads(text)
        if isinstance(data, dict) and "selected_tags" in data:
            return data["selected_tags"]
        if isinstance(data, list):
            return data
    except Exception:
        pass

    # 텍스트 안에서 리스트만 추출
    match = re.search(r"\[[\s\S]*\]", text)
    if match:
        list_text = match.group(0)
        try:
            return json.loads(list_text)
        except Exception:
            try:
                return ast.literal_eval(list_text)
            except Exception:
                pass

    return []


def extract_tags_with_qwen(query_image_path, all_tags):
    image_path = resolve_path(query_image_path)
    tag_candidates = ", ".join(all_tags)

    prompt = f"""
너는 이미지 태깅 모델이다.

입력 이미지를 보고, 아래 [태그 후보 목록] 중 실제 이미지에 해당하는 태그만 골라라.

절대 규칙:
1. 반드시 [태그 후보 목록]에 있는 태그만 선택한다.
2. 후보 목록에 없는 새 태그를 만들지 않는다.
3. 이미지에 사람 1명만 있으면 사람_1만 선택하고, 사람_2/사람_3/사람_4 관련 태그는 절대 선택하지 않는다.
4. 이미지에 사람 2명 이상이 보일 때만 사람_2, 사람_3, 사람_4 관련 태그를 선택한다.
5. 사람이 카메라를 바라보거나 얼굴/몸이 정면이면 사람_1_정면을 선택한다.
6. 사람이 옆을 보고 있으면 사람_1_측면을 선택한다.
7. 사람이 뒤돌아 있으면 사람_1_후면을 선택한다.
8. 바다가 보이면 바다를 선택한다.
9. 바닥, 도로, 땅, 보도블록, 콘크리트 바닥이 보이면 땅을 선택한다.
10. 하늘에 구름이 보이면 구름을 선택한다.
11. 낮 시간대이면 낮을 선택한다.
12. 밤이면 밤을 선택한다.
13. 해질녘/붉은 하늘/석양이면 노을 또는 태양을 선택한다.
14. 맑은 날씨이면 맑음을 선택한다.
15. 흐린 날씨이면 흐림을 선택한다.
16. 바다 위 또는 바다 배경의 상생의 손 조형물이 보이면 상생의손_바다를 선택한다.
17. 육지/광장 위의 상생의 손 조형물이 보이면 상생의손_육지를 선택한다.
18. 사람이 상생의 손과 비슷하게 손을 들어 포즈를 취하면 가짜손포즈를 선택한다.
19. 결과는 JSON만 출력한다.

[태그 후보 목록]
{tag_candidates}

출력 형식:
{{
  "selected_tags": ["바다", "사람_1", "사람_1_정면"]
}}
"""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(qwen_model.device)

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    selected = parse_json_list_from_text(output_text)

    all_tag_set = set(all_tags)
    selected = [str(tag).strip() for tag in selected if str(tag).strip() in all_tag_set]

    return selected, output_text


# =========================
# 6. RAG 필터링
# =========================

def tag_match_score(query_tags, item_tags):
    query_set = set(query_tags)
    item_set = set([str(t).strip() for t in item_tags])
    matched = sorted(list(query_set & item_set))
    return len(matched), matched


# =========================
# 7. 전체 파이프라인 실행
# =========================

def run_pipeline():
    database = load_database()
    query_image_path = get_query_image_path()

    print("=" * 80)
    print("기준 사진")
    print("=" * 80)
    print(query_image_path)
    display(Image.open(resolve_path(query_image_path)))

    all_tags = collect_all_tags(database)

    print("=" * 80)
    print("tags.txt 후보 태그")
    print("=" * 80)
    print(", ".join(all_tags))

    # 1) Qwen3-VL 태그 선택
    filter_tags, qwen_raw_output = extract_tags_with_qwen(query_image_path, all_tags)

    print("=" * 80)
    print("Qwen3-VL이 선택한 필터 태그")
    print("=" * 80)
    print(", ".join(filter_tags) if filter_tags else "선택된 태그 없음")

    print("=" * 80)
    print("Qwen3-VL 원본 출력")
    print("=" * 80)
    print(qwen_raw_output)

    # 2) 기준 사진 CLIP 이미지 임베딩
    query_embedding = get_image_embedding(query_image_path)

    # 3) RAG 필터링
    passed = []
    excluded = []

    for item in database:
        match_count, matched_tags = tag_match_score(filter_tags, item["tags"])

        if match_count >= MIN_FILTER_MATCH:
            image_sim = cosine_similarity(query_embedding, item["image_embedding"])

            passed.append({
                "filename": item["filename"],
                "image_path": item["image_path"],
                "group": item["group"],
                "tags": item["tags"],
                "matched_tags": matched_tags,
                "filter_match_count": match_count,
                "image_similarity": image_sim,
            })
        else:
            excluded.append({
                "filename": item["filename"],
                "image_path": item["image_path"],
                "group": item["group"],
                "tags": item["tags"],
                "matched_tags": matched_tags,
                "filter_match_count": match_count,
                "reason": "Qwen3-VL 선택 태그와 매칭 부족",
            })

    # 4) CLIP 이미지 유사도 순위화
    ranked = sorted(passed, key=lambda x: x["image_similarity"], reverse=True)

    print("=" * 80)
    print("RAG 필터링 결과")
    print("=" * 80)
    print(f"통과 사진 수: {len(passed)}")
    print(f"제외 사진 수: {len(excluded)}")

    print("=" * 80)
    print(f"TOP-{TOP_K} 유사 사진 순위")
    print("=" * 80)

    for idx, item in enumerate(ranked[:TOP_K], start=1):
        print(f"\n{idx}위")
        print(f"파일명: {item['filename']}")
        print(f"그룹: {item['group']}")
        print(f"이미지 유사도: {item['image_similarity']:.4f}")
        print(f"매칭 태그 수: {item['filter_match_count']}")
        print(f"매칭 태그: {', '.join(item['matched_tags'])}")
        print(f"경로: {item['image_path']}")
        display(Image.open(resolve_path(item["image_path"])))

    print("=" * 80)
    print("제외된 사진")
    print("=" * 80)

    for idx, item in enumerate(excluded, start=1):
        print(f"\n제외 {idx}")
        print(f"파일명: {item['filename']}")
        print(f"그룹: {item['group']}")
        print(f"매칭 태그 수: {item['filter_match_count']}")
        print(f"매칭 태그: {', '.join(item['matched_tags']) if item['matched_tags'] else '없음'}")
        print(f"이유: {item['reason']}")
        display(Image.open(resolve_path(item["image_path"])))

    return ranked, excluded, filter_tags, qwen_raw_output


ranked, excluded, filter_tags, qwen_raw_output = run_pipeline()

device: cuda
CLIP 모델 로드 중...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 15347.37it/s]


CLIP 모델 로드 완료
Qwen-VL 모델 로드 중...


c:\ProgramData\anaconda3\envs\qwen3vl\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\김동준\.cache\huggingface\hub\models--Qwen--Qwen2.5-VL-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files:   0%|          | 0/2 [10:39<?, ?it/s]


KeyboardInterrupt: 